# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os, getpass, duckdb, pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
ANCHOR = "DATE '2026-03-31'"

features = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date >  {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
        AVG(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_avg_position END)        AS pos_prev30
    FROM {fact} f
    WHERE f.report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2 HAVING imp_prev30 >= 100
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']
features = features.merge(
    con.sql(f"""SELECT content_hash_id, DATE_DIFF('day', content_created_date, {ANCHOR}) AS content_age_days
                FROM {dim_content}""").df(),
    on='content_hash_id', how='left'
)
features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)
features = features.dropna(subset=['pos_prev30', 'content_age_days']).reset_index(drop=True)

FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30', 'content_age_days']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(features, groups=features['client_hash_id']))
train, test = features.iloc[train_idx].copy(), features.iloc[test_idx].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(train[FEATURE_COLS], train['is_declining'])
test['model_score'] = rf.predict_proba(test[FEATURE_COLS])[:, 1]

stale = (test['content_age_days'] >= 365).astype(int)
visible = (test['imp_prev30'] >= 250).astype(int)
test['reason_code'] = np.where((stale & visible) == 1, 'stale_but_visible', 'model_flagged_only')

queue = test.sort_values('model_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1
print(queue[['rank','client_hash_id','content_hash_id','model_score','reason_code']].head(10))

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rank           client_hash_id           content_hash_id  model_score  \
0     1  client_62f4a7e64f5e0096  content_5e3c59275a112354     0.655377   
1     2  client_62f4a7e64f5e0096  content_fdb05774546c8909     0.654912   
2     3  client_62f4a7e64f5e0096  content_1dcdc75d56ca1b21     0.654269   
3     4  client_62f4a7e64f5e0096  content_84dc91f3a2eeb628     0.653478   
4     5  client_62f4a7e64f5e0096  content_848bdcfa10ccacd1     0.653267   
5     6  client_62f4a7e64f5e0096  content_965cf8df4b6e4228     0.653115   
6     7  client_62f4a7e64f5e0096  content_f2c87dcda8cb44f7     0.652565   
7     8  client_62f4a7e64f5e0096  content_61d2ced86b23a928     0.652565   
8     9  client_62f4a7e64f5e0096  content_e564e51ec78d4b24     0.652443   
9    10  client_62f4a7e64f5e0096  content_f471eeda89350fa6     0.652151   

          reason_code  
0  model_flagged_only  
1  model_flagged_only  
2  model_flagged_only  
3  model_flagged_only  
4  model_flagged_only  
5  model_flagged_only  
6  mod

Each row is ranked by the model's predicted probability of decline, with a reason code noting whether it also matches the simpler age/visibility rule from ML-07 (**stale_but_visible**) or was flagged by the model alone **model_flagged_only**). This is a shortlist for review, not a validated priority order — see Section 2 for why that distinction matters.

This run's top 10 also demonstrates the risk named in Section 2 directly: all ten items belong to a single client, and all ten are **model_flagged_only** rather than **stale_but_visible**. That means the highest-ranked items are simultaneously the least independently verifiable and the most likely to reflect one client's scale rather than genuine decline signal — exactly why no action here should proceed without a human checking the underlying feature values first.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a starting shortlist for a content strategist to review manually, alongside judgment and context the model doesn't have (seasonality, business priority, recent site changes). Decision-support, not a decision.

**The limit that matters most:** on a held-out set of clients the model never saw during training, **precision@50** measured at 0.200, below the 0.279 base rate. Pulling 50 items at random from this population would, on average, contain more genuinely declining pages than the model's top 50 did in this test. The model has not been shown to outperform random selection at this threshold.

**Where this might still have value:** the stale_but_visible reason code (**age** ≥ 365 days AND **prior-30-day impressions** ≥ 250) is a simple, transparent rule a human can verify by eye, unlike the model's score, its reasoning is inspectable at a glance.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Required human review:** every flagged item must be manually reviewed before any action, not a spot-check sample. The model hasn't cleared the base rate, so treating its output as pre-validated would be dishonest.

**What must NOT be automated:**
- No automatic publishing, rewriting, or deletion of content based on this score
- No automatic reduction in marketing/dev budget tied to a "declining" label
- The score must never be shown to a client as a certainty metric
- No action on model_flagged_only items without a human checking the underlying
  imp_prev30, pos_prev30, and content_age_days values directly

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- Re-run the **precision@50**-vs-base-rate check quarterly on a fresh, unseen set of clients; if still at or below base rate, this playbook should not be presented as validated in whatever form it takes next
- If the client roster changes substantially, do not trust this ranking without
  re-validating — ML-09 showed this model can memorize client-specific scale patterns rather than genuine decline signal
- Monitor the base rate itself; if it shifts meaningfully, the comparison point moves and needs re-establishing

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
import os, json

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue.to_csv('work/outputs/content_action_playbook_queue.csv', index=False)

metrics = {
    'precision_at_50': 0.200,
    'base_rate': 0.279,
    'naive_split_precision_at_50': 0.92,
    'grouped_split_precision_at_50': 0.20,
    'note': 'Model did not exceed base rate on grouped/honest split. Decision-support only.'
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported queue and metrics for the paper")

Exported queue and metrics for the paper


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.